In [2]:
import torch
print("Number of GPUs Present:", torch.cuda.device_count())
print("GPU Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

Number of GPUs Present: 0
GPU Available: False
Device Name: No GPU detected


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
import joblib

/csl/users/2026nnandaku/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# -------------------------
# 1. Load dataset
# -------------------------
df = pd.read_csv('PI1M_with_Tg_Final.csv')
df = df[df['Tg'].notna()].copy()

In [4]:
# -------------------------
# 2. Subsample for tuning
# -------------------------
df_sample = df.sample(n=75000, random_state=42)

In [5]:
# -------------------------
# 3. Split features and target
# -------------------------
X_sample = df_sample.drop(columns=['SMILES', 'Name', 'Tg', 'Tg_LF', 'Tg_predicted'])
y_sample = df_sample['Tg']

X_train, X_val, y_train, y_val = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42
)

In [ ]:
# -------------------------
# 4. GridSearchCV on sample
# -------------------------
param_grid = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [15, 20, 25, 30],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 4]
}

rmse_scorer = make_scorer(mean_squared_error, greater_is_better=False, squared=False)

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    scoring=rmse_scorer,
    cv=5,
    n_jobs=-1,
    verbose=3
)

grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_

print("\nGrid Search Complete. Best Params:")
print(best_params)

Fitting 5 folds for each of 144 candidates, totalling 720 fits
[CV 5/5] END max_depth=15, min_samples_leaf=1, min_samples_split=2, n_estimators=200;, score=-35.887 total time= 3.8min
[CV 2/5] END max_depth=15, min_samples_leaf=1, min_samples_split=2, n_estimators=500;, score=-35.879 total time= 9.6min


/csl/users/2026nnandaku/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/csl/users/2026nnandaku/.local/lib/python3.8/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 3/5] END max_depth=15, min_samples_leaf=1, min_samples_split=2, n_estimators=200;, score=-36.007 total time= 3.9min
[CV 4/5] END max_depth=15, min_samples_leaf=1, min_samples_split=2, n_estimators=500;, score=-35.587 total time= 9.8min


/csl/users/2026nnandaku/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


[CV 1/5] END max_depth=15, min_samples_leaf=1, min_samples_split=2, n_estimators=200;, score=-35.913 total time= 3.9min
[CV 3/5] END max_depth=15, min_samples_leaf=1, min_samples_split=2, n_estimators=500;, score=-35.962 total time= 9.8min
[CV 4/5] END max_depth=15, min_samples_leaf=1, min_samples_split=6, n_estimators=200;, score=-35.613 total time= 3.8min
[CV 5/5] END max_depth=15, min_samples_leaf=1, min_samples_split=6, n_estimators=400;, score=-35.786 total time= 7.6min
[CV 3/5] END max_depth=15, min_samples_leaf=2, min_samples_split=2, n_estimators=400;, score=-35.887 total time= 7.5min
[CV 5/5] END max_depth=15, min_samples_leaf=2, min_samples_split=4, n_estimators=300;, score=-35.811 total time= 5.7min
[CV 1/5] END max_depth=15, min_samples_leaf=2, min_samples_split=6, n_estimators=300;, score=-35.807 total time= 5.6min
[CV 3/5] END max_depth=15, min_samples_leaf=2, min_samples_split=6, n_estimators=500;, score=-35.879 total time= 9.3min
[CV 5/5] END max_depth=15, min_samples_l

/csl/users/2026nnandaku/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


[CV 5/5] END max_depth=15, min_samples_leaf=1, min_samples_split=2, n_estimators=300;, score=-35.852 total time= 5.8min
[CV 4/5] END max_depth=15, min_samples_leaf=1, min_samples_split=4, n_estimators=200;, score=-35.615 total time= 3.8min
[CV 1/5] END max_depth=15, min_samples_leaf=1, min_samples_split=4, n_estimators=500;, score=-35.859 total time= 9.5min
[CV 2/5] END max_depth=15, min_samples_leaf=1, min_samples_split=6, n_estimators=500;, score=-35.832 total time= 9.4min
[CV 4/5] END max_depth=15, min_samples_leaf=2, min_samples_split=2, n_estimators=500;, score=-35.507 total time= 9.3min
[CV 1/5] END max_depth=15, min_samples_leaf=2, min_samples_split=6, n_estimators=200;, score=-35.838 total time= 3.7min
[CV 5/5] END max_depth=15, min_samples_leaf=2, min_samples_split=6, n_estimators=300;, score=-35.803 total time= 5.5min
[CV 1/5] END max_depth=15, min_samples_leaf=4, min_samples_split=2, n_estimators=300;, score=-35.786 total time= 5.3min
[CV 3/5] END max_depth=15, min_samples_l

/csl/users/2026nnandaku/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


[CV 4/5] END max_depth=15, min_samples_leaf=1, min_samples_split=2, n_estimators=400;, score=-35.603 total time= 7.7min
[CV 5/5] END max_depth=15, min_samples_leaf=1, min_samples_split=4, n_estimators=300;, score=-35.830 total time= 5.7min
[CV 2/5] END max_depth=15, min_samples_leaf=1, min_samples_split=6, n_estimators=200;, score=-35.870 total time= 3.8min
[CV 3/5] END max_depth=15, min_samples_leaf=1, min_samples_split=6, n_estimators=400;, score=-35.929 total time= 7.6min
[CV 5/5] END max_depth=15, min_samples_leaf=2, min_samples_split=2, n_estimators=300;, score=-35.811 total time= 5.6min
[CV 5/5] END max_depth=15, min_samples_leaf=2, min_samples_split=4, n_estimators=200;, score=-35.832 total time= 3.8min
[CV 1/5] END max_depth=15, min_samples_leaf=2, min_samples_split=4, n_estimators=500;, score=-35.783 total time= 9.4min
[CV 2/5] END max_depth=15, min_samples_leaf=2, min_samples_split=6, n_estimators=500;, score=-35.735 total time= 9.2min
[CV 4/5] END max_depth=15, min_samples_l

/csl/users/2026nnandaku/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [ ]:
# -------------------------
# 5. Train final model on full data using best params
# -------------------------
X_full = df.drop(columns=['SMILES', 'Name', 'Tg', 'Tg_LF', 'Tg_predicted'])
y_full = df['Tg']

final_model = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_full, y_full)

In [ ]:
# -------------------------
# 6. Evaluate on holdout test set
# -------------------------
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)
y_pred = final_model.predict(X_test_full)

mae = mean_absolute_error(y_test_full, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_full, y_pred))
r2 = r2_score(y_test_full, y_pred)

print(f"\nFinal Model MAE:  {mae:.3f}")
print(f"Final Model RMSE: {rmse:.3f}")
print(f"Final Model R²:   {r2:.3f}")

In [ ]:
# -------------------------
# 7. Save final model
# -------------------------
joblib.dump(final_model, 'tg_predictor_rf_final.pkl')